# MS2 Training — Arabic NLP QA RAG

**Before running:** Runtime → Change runtime type → **T4 GPU**

Cells run top to bottom. MS1 pipeline (~2 min) → MS2 data prep (~3 min) → Train A (~15 min) → Train B (~15 min) → Evaluate & compare.

In [ ]:
# Cell 1: Clone repo and install dependencies
!git clone https://github.com/seif495/arabic-nlp-qa-rag
%cd arabic-nlp-qa-rag
!pip install sentencepiece tensorflow -q

In [ ]:
# Cell 2: Verify GPU is available
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Cell 3: Run MS1 pipeline — cleans transcripts and builds processed JSONL
!python -m src.cli.ms1 run-all

In [ ]:
# Cell 4: Verify MS1 output exists
import os
path = 'data/processed/ms1/ms1_dataset_processed_v001.jsonl'
size = os.path.getsize(path)
print(f'MS1 dataset: {path} ({size:,} bytes)')

# Count records
with open(path, encoding='utf-8') as f:
    n = sum(1 for _ in f)
print(f'Records: {n}')

In [ ]:
# Cell 5: MS2 data prep — trains BPE tokenizer, builds char vocab, writes TFRecords
!python -m src.cli.ms2 prep-data

In [ ]:
# Cell 6: Write short-budget run configs (15 min each — enough for real numbers)
import json

base = {
    'l_q': 32, 'l_c': 284, 'l_enc': 320, 'l_dec': 64,
    'bucket_boundaries': [128, 192, 256, 320],
    'target_tokens_per_batch': 16384,
    'wall_clock_budget_minutes': 15,
    'label_smoothing': 0.1,
    'gradient_clip_norm': 1.0,
    'seed': 42,
}

config_a = {**base, 'model_id': 'A', 'lr_schedule': 'cosine_with_warmup'}
config_b = {**base, 'model_id': 'B', 'lr_schedule': 'noam'}

with open('config_a.json', 'w') as f:
    json.dump(config_a, f, indent=2)
with open('config_b.json', 'w') as f:
    json.dump(config_b, f, indent=2)

print('config_a.json:', json.dumps(config_a, indent=2))
print('config_b.json:', json.dumps(config_b, indent=2))

In [ ]:
# Cell 7: Train Model A (RNN + gated merge + FiLM) — ~15 min
!python -m src.cli.ms2 train --model a --seed 42 --config config_a.json

In [ ]:
# Cell 8: Train Model B (Transformer + RoPE) — ~15 min
!python -m src.cli.ms2 train --model b --seed 42 --config config_b.json

In [ ]:
# Cell 9: Run inference on dev set for both models
!python -m src.cli.ms2 infer --model a --seed 42
!python -m src.cli.ms2 infer --model b --seed 42

In [ ]:
# Cell 10: Evaluate both models
!python -m src.cli.ms2 evaluate --model a --seed 42
!python -m src.cli.ms2 evaluate --model b --seed 42

In [ ]:
# Cell 11: Build headline comparison table
!python -m src.cli.ms2 compare

In [ ]:
# Cell 12: Print the headline table
with open('docs/reports/ms2_headline_table.md', encoding='utf-8') as f:
    print(f.read())

In [ ]:
# Cell 13: Download all experiment artifacts as a zip
import shutil
shutil.make_archive('ms2_results', 'zip', '.', 'experiments/ms2')
shutil.make_archive('ms2_reports', 'zip', '.', 'docs/reports')

from google.colab import files
files.download('ms2_results.zip')
files.download('ms2_reports.zip')